> The monitoring data and building-specific configuration are confidential and are not included. See `README.md` for execution requirements. Clear any generated outputs before sharing an executed notebook.


In [ ]:
# Private workspace and configuration
from copy import deepcopy as _privacy_deepcopy
from functools import lru_cache as _privacy_lru_cache
import json
import os
from pathlib import Path


def _private_config_path(variable):
    value = os.environ.get(variable)
    if not value:
        raise RuntimeError(f"Set {variable} before running this notebook; see README.md.")
    return Path(value).expanduser().resolve()


def private_workspace():
    """Return the configured private input/output directory."""
    path = _private_config_path("HVAC_PRIVATE_WORKSPACE")
    if not path.is_dir():
        raise FileNotFoundError("The configured private workspace does not exist.")
    return path


def _decode_private_values(value):
    if isinstance(value, list):
        return [_decode_private_values(item) for item in value]
    if isinstance(value, dict):
        if set(value) == {"__tuple_key_dict__"}:
            return {tuple(key): _decode_private_values(item)
                    for key, item in value["__tuple_key_dict__"]}
        return {key: _decode_private_values(item) for key, item in value.items()}
    return value


@_privacy_lru_cache(maxsize=1)
def _load_private_values():
    with _private_config_path("HVAC_PRIVATE_SETTINGS").open(encoding="utf-8") as handle:
        values = json.load(handle).get("private_values")
    if not isinstance(values, dict):
        raise ValueError("Private settings must contain a private_values object.")
    return _decode_private_values(values)


def private_value(key):
    """Read a private setting without embedding its value in the public source."""
    values = _load_private_values()
    if key not in values or values[key] is None:
        raise KeyError(f"Required private setting is missing: {key!r}.")
    return _privacy_deepcopy(values[key])


# Notebook 06: Duration-calculation and endpoint sensitivity

This notebook performs two separate analyses using the trained models, split and recorded events from Notebooks 01–03. Model parameters remain fixed.

**Duration calculation:** Hold the neural and Temperature ET forecasts fixed while varying final-trend extrapolation settings: fitted point count, minimum slope, extrapolation cap and extrapolation on/off. Direct ET remains an unchanged reference. Reproduce the saved evaluation predictions before applying these perturbations.

**Observed endpoints:** Hold the original event cohort and starts fixed, then recompute endpoints with temperature allowances of 0.3, 0.4 and 0.5 degrees Celsius. The default 0.4-degree rule must reproduce the saved endpoints before alternative allowances are analyzed.

The first analysis measures sensitivity of predictions to the tested extrapolation settings. The second measures sensitivity of observed duration labels to their definition; it neither retrains models nor re-scores their accuracy against alternative labels. Neither analysis selects replacement settings from test performance.

In stored tables, `phase` denotes operating mode and an anchor identifies a duration prediction time. Private observations and configuration are required as described in `README.md`.


In [ ]:


# Imports, exact reference paths, and constants

import os, re, json, math, hashlib, warnings, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    import torch
    import torch.nn as nn
except Exception as e:
    raise RuntimeError('PyTorch is required for Notebook 06.') from e

try:
    from google.colab import drive  # type: ignore
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

warnings.filterwarnings('default')
# Keep scientific warnings visible while suppressing a repeated client-library
# deprecation warning.
warnings.filterwarnings(
    'ignore',
    message=r'datetime\.datetime\.utcnow\(\) is deprecated.*',
    category=DeprecationWarning,
)

def safe_torch_load(path, map_location=None):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

# AUTHORITATIVE reference directories

BASE_DIR = Path(str(private_workspace()))
if 'PAPER-TEMP-NEW' in str(BASE_DIR):
    raise RuntimeError('STOP: Notebook 06 must use PAPER-TEMP, not PAPER-TEMP-NEW.')

DATA_DIR = BASE_DIR / 'full_rampup_5min_feature_dataset_UNIFIED_DERIVATIVE_DURATION_AWARE'

TRAIN_RUN = (
    BASE_DIR /
    'training_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL' /
    'paper_final_q_weighted'
)
TRAIN_MODEL_DIR = TRAIN_RUN / 'models'

NOTEBOOK03_RUN_ID = private_value('shared.notebook03_run_id')
NB03_RUN = (
    BASE_DIR /
    'duration_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL' /
    NOTEBOOK03_RUN_ID
)
NB03_PRED_PATH = (
    NB03_RUN /
    'predictions' /
    'ALL_remaining_duration_predictions_with_relative_errors.csv'
)
NB03_MAIN_PATH = (
    NB03_RUN /
    'tables' /
    'PAPER_MAIN_duration_metrics_EXACT_COMMON_SUPPORT.csv'
)
NB03_MANIFEST_PATH = (
    NB03_RUN /
    'duration_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL.json'
)

OUT_ROOT = (
    BASE_DIR /
    'sensitivity_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL' /
    'paper_final_converter_and_event_end'
)
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
PRED_DIR = OUT_ROOT / 'predictions'
AUDIT_DIR = OUT_ROOT / 'audit'
CACHE_DIR = OUT_ROOT / 'cache'
for p in [TABLE_DIR, FIG_DIR, PRED_DIR, AUDIT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Exact Notebook-03 reference constants

PHASES = ['heating', 'cooling']
STEP_MIN = 5.0
END_BAND_C = 0.4

INDIRECT_MAX_HORIZON_BY_PHASE = {
    'heating': 185.0,
    'cooling': 65.0,
}

CANONICAL_CONVERTER = {
    'final_n_points': 4,
    'min_abs_slope_C_per_min': 1e-4,
    'max_extra_min_by_phase': {
        'heating': 60.0,
        'cooling': 30.0,
    },
    'allow_extrapolation': True,
}

DIRECT_METHOD = 'Direct_RoomPhase_ET'
TRAJECTORY_METHODS = [
    'Indirect_RoomPhase_ET',
    'Unified_Derivative_NN',
]
PAPER_METHODS = [
    DIRECT_METHOD,
    *TRAJECTORY_METHODS,
]

# Event-end sensitivity

END_BANDS_C = [0.3, 0.4, 0.5]
CANONICAL_EVENT_END_BAND_C = 0.4

MIN_RAMPUP_MINUTES = 5.0
MAX_RAMPUP_MINUTES_BY_PHASE = {
    'heating': 180.0,
    'cooling': 60.0,
}
PHASE_END_SEARCH_WINDOWS = {
    'heating': ('06:00', '10:00'),
    'cooling': ('05:00', '09:30'),
}

# Bootstrap

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42
CI_LEVEL = 0.95

# Runtime controls.
REUSE_TRAJECTORY_CACHE = True
STRICT_CANONICAL_REPLAY = True
CANONICAL_REPLAY_TOL_MIN = 1e-3
STRICT_EVENT_END_REPRODUCTION = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.set_float32_matmul_precision('high')

print('DEVICE:', DEVICE)
print('DATA_DIR:', DATA_DIR)
print('TRAIN_RUN:', TRAIN_RUN)
print('Notebook 03 run:', NB03_RUN)
print('Notebook 06 output:', OUT_ROOT)


## 1. Reconstruct the evaluation cases

Load the duration prediction keys, input-time gaps, observed labels and chronological split used in Notebook 03. Verify split fingerprints against the saved training and evaluation records, then reconstruct the common test cases.


In [ ]:


# Saved anchors, shared split, and exact-common support

def norm_event_id(x):
    if pd.isna(x):
        raise RuntimeError('Missing event_id')
    s = str(x).strip()
    if re.fullmatch(r'[-+]?\d+\.0+', s):
        s = str(int(float(s)))
    return s

def shared_split_sha256(path):
    z = pd.read_csv(path)
    z['room'] = z['room'].astype(str).str.strip()
    z['event_id'] = z['event_id'].map(norm_event_id)
    cols = ['phase', 'room', 'event_id', 'split', 'split_block']
    z = (
        z[cols]
        .drop_duplicates()
        .sort_values(['phase', 'room', 'event_id'])
        .reset_index(drop=True)
    )
    payload = z.to_csv(index=False, lineterminator='\n').encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

SPLIT_PATH = DATA_DIR / 'shared_event_split_map.csv'
CURRENT_SPLIT_SHA256 = shared_split_sha256(SPLIT_PATH)

split = pd.read_csv(SPLIT_PATH)
split['event_id'] = split['event_id'].map(norm_event_id)
split['room'] = split['room'].astype(str).str.strip()
split['event_key'] = (
    split['phase'].astype(str) + '|' +
    split['room'] + '|' +
    split['event_id']
)
split = split[
    ['event_key', 'split', 'split_block']
].drop_duplicates()

frames = []
for phase in PHASES:
    p = DATA_DIR / phase / f'{phase}_remaining_duration_anchors_5min.csv'
    if not p.exists():
        raise FileNotFoundError(p)

    d = pd.read_csv(p)
    d['phase'] = phase

    for c in [
        'timestamp', 'anchor_timestamp', 'feature_source_timestamp',
        'rampup_start_time', 'rampup_end_time', 'rampup_end_grid_time'
    ]:
        if c in d.columns:
            d[c] = pd.to_datetime(d[c], errors='coerce')

    d['event_id'] = d['event_id'].map(norm_event_id)
    d['room'] = d['room'].astype(str).str.strip()
    d['event_key'] = (
        d['phase'] + '|' +
        d['room'] + '|' +
        d['event_id']
    )

    d = d.merge(
        split,
        on='event_key',
        how='left',
        validate='many_to_one'
    )
    if d['split'].isna().any():
        raise RuntimeError(f'{phase}: unmatched duration anchors.')

    d['true_remaining_duration_min'] = pd.to_numeric(
        d.get('true_remaining_duration_min', d.get('remaining_duration_min')),
        errors='coerce'
    )
    d['remaining_trajectory_quality_score'] = pd.to_numeric(
        d['remaining_trajectory_quality_score'],
        errors='coerce'
    ).clip(0.0, 1.0)

    d['asof_gap_min'] = (
        d['anchor_timestamp'] - d['feature_source_timestamp']
    ).dt.total_seconds() / 60.0
    d['asof_gap_min'] = d['asof_gap_min'].clip(lower=0.0)

    kind = (
        d['anchor_kind']
        if 'anchor_kind' in d.columns
        else pd.Series('anchor', index=d.index)
    ).astype(str)
    d['anchor_kind'] = kind

    d['anchor_key'] = (
        d['event_key'] + '|' +
        d['anchor_timestamp'].astype(str) + '|' +
        kind
    )

    d['thermal_mode_sign'] = pd.to_numeric(
        d['thermal_mode_sign'], errors='coerce'
    )

    d['Duration_Crossing_Threshold_C'] = (
        pd.to_numeric(d['Phase_Target_T_C'], errors='coerce') -
        d['thermal_mode_sign'] * END_BAND_C
    )

    frames.append(d)

ANCHORS = pd.concat(frames, ignore_index=True, sort=False)

if ANCHORS['anchor_key'].duplicated().any():
    raise RuntimeError('Authoritative anchor_key is not unique.')

EVAL_ANCHORS = (
    ANCHORS[ANCHORS['split'].eq('test')]
    .copy()
    .reset_index(drop=True)
)

for required in [
    TRAIN_RUN / 'training_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL.json',
    NB03_MANIFEST_PATH,
    NB03_PRED_PATH,
    NB03_MAIN_PATH,
]:
    if not required.exists():
        raise FileNotFoundError(required)

with open(
    TRAIN_RUN / 'training_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL.json',
    'r',
    encoding='utf-8'
) as f:
    TRAIN_MANIFEST = json.load(f)

with open(NB03_MANIFEST_PATH, 'r', encoding='utf-8') as f:
    NB03_MANIFEST = json.load(f)

if TRAIN_MANIFEST.get('shared_split_sha256') != CURRENT_SPLIT_SHA256:
    raise RuntimeError('Notebook 02 shared split fingerprint mismatch.')
if NB03_MANIFEST.get('shared_split_sha256') != CURRENT_SPLIT_SHA256:
    raise RuntimeError('Notebook 03 shared split fingerprint mismatch.')

NB03_ALL = pd.read_csv(NB03_PRED_PATH)
NB03_TEST = NB03_ALL[NB03_ALL['split'].eq('test')].copy()

wide = NB03_TEST.pivot_table(
    index='anchor_key',
    columns='method',
    values='pred_remaining_duration_min',
    aggfunc='first'
)
for m in PAPER_METHODS:
    if m not in wide.columns:
        wide[m] = np.nan

EXACT_COMMON_KEYS = set(
    wide.dropna(subset=PAPER_METHODS).index.astype(str)
)

COMMON_ANCHORS = EVAL_ANCHORS[
    EVAL_ANCHORS['anchor_key'].astype(str).isin(EXACT_COMMON_KEYS)
].copy()

support_summary = (
    EVAL_ANCHORS.groupby('phase')
    .agg(
        full_test_anchors=('anchor_key', 'nunique'),
        full_test_events=('event_key', 'nunique'),
        full_test_rooms=('room', 'nunique'),
    )
    .reset_index()
    .merge(
        COMMON_ANCHORS.groupby('phase')
        .agg(
            exact_common_anchors=('anchor_key', 'nunique'),
            exact_common_events=('event_key', 'nunique'),
            exact_common_rooms=('room', 'nunique'),
        )
        .reset_index(),
        on='phase',
        how='left'
    )
)
support_summary['retained_anchor_pct'] = (
    100.0 *
    support_summary['exact_common_anchors'] /
    support_summary['full_test_anchors']
)

support_summary.to_csv(
    TABLE_DIR / 'NB06_exact_common_support_summary.csv',
    index=False
)

print('Shared split SHA256:', CURRENT_SPLIT_SHA256)
display(support_summary)


## 2. Load trained trajectory models

Load the Shared NN, room/mode Temperature ET models and saved transformations from Notebook 02. Apply the input transformations used by Notebook 03 without fitting new models.


In [ ]:


# Fixed model artifacts and exact Notebook-03 transforms

PREP = joblib.load(
    TRAIN_MODEL_DIR / 'unified_numeric_preprocessor.joblib'
)
MODEL_INPUTS = list(PREP['numeric_cols'])

class UnifiedDirectedRateNN(nn.Module):
    def __init__(
        self,
        n_in,
        hidden=(128, 64, 32),
        dropout=0.10
    ):
        super().__init__()
        layers = []
        prev = n_in
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.Tanh()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

ckpt = safe_torch_load(
    TRAIN_MODEL_DIR / 'unified_directed_rate_nn.pt',
    map_location=DEVICE
)
if ckpt.get('shared_split_sha256') != CURRENT_SPLIT_SHA256:
    raise RuntimeError('Canonical NN checkpoint split mismatch.')

NN_MODEL = UnifiedDirectedRateNN(
    ckpt['n_inputs'],
    ckpt['hidden_dims'],
    ckpt['dropout']
).to(DEVICE)
NN_MODEL.load_state_dict(ckpt['state_dict'])
NN_MODEL.eval()

ET_ROOT = TRAIN_MODEL_DIR / 'room_phase_extra_trees_temperature'
ET_MANIFEST_PATH = ET_ROOT / 'model_manifest.csv'
if not ET_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Canonical room/phase temperature ET manifest missing: {ET_MANIFEST_PATH}'
    )

ET_ART = {}
et_manifest = pd.read_csv(ET_MANIFEST_PATH)

for _, r in et_manifest.iterrows():
    phase = str(r['phase'])
    room = str(r['room'])

    p = Path(r['artifact_path'])
    if not p.exists():
        p = ET_ROOT / phase / f'{room}.joblib'

    if not p.exists():
        raise FileNotFoundError(
            f'Missing canonical temperature ET artifact: {phase}/{room}: {p}'
        )

    art = joblib.load(p)

    for k in ['model', 'imputer', 'feature_names']:
        if k not in art:
            raise RuntimeError(
                f'{phase}/{room}: ET artifact missing key {k}'
            )

    ET_ART[(phase, room)] = art

def transform_nn(df):
    x = (
        df[MODEL_INPUTS]
        .apply(pd.to_numeric, errors='coerce')
        .replace([np.inf, -np.inf], np.nan)
        .copy()
    )

    phases = df['phase'].astype(str).to_numpy()

    for phase in PHASES:
        idx = np.where(phases == phase)[0]
        if len(idx) == 0:
            continue

        med = PREP['phase_medians'][phase]

        for c in MODEL_INPUTS:
            j = x.columns.get_loc(c)
            x.iloc[idx, j] = (
                x.iloc[idx, j]
                .fillna(float(med[c]))
                .to_numpy()
            )

    if x.isna().any().any():
        raise RuntimeError('NN transform left NaN values.')

    return PREP['scaler'].transform(
        x.to_numpy(float)
    ).astype(np.float32)

print('Loaded canonical NN epoch:', ckpt.get('best_epoch'))
print('Loaded room/phase temperature ET models:', len(ET_ART))


## 3. Parameterize the evaluation duration calculation

Retain the current-condition and forecast-crossing rules from Notebook 03. Expose only the final-trend extrapolation parameters for sensitivity analysis. Other choices, including forecast-grid resolution and crossing conditions, remain fixed.


In [ ]:


# Parameterized Notebook-03 evaluation completion-time procedure

def crossed(phase, T, threshold):
    return (
        float(T) >= float(threshold)
        if phase == 'heating'
        else float(T) <= float(threshold)
    )

def direction_valid(phase, dT):
    return (
        np.isfinite(dT) and
        (
            (dT > 0)
            if phase == 'heating'
            else (dT < 0)
        )
    )

def linear_crossing(h0, T0, h1, T1, threshold):
    if abs(T1 - T0) < 1e-12:
        return float(h1)
    a = np.clip(
        (threshold - T0) / (T1 - T0),
        0, 1
    )
    return float(h0 + a * (h1 - h0))

def estimate_crossing_from_source_parametric(
    phase,
    current_T,
    threshold,
    horizons,
    pred_T,
    current_delta_prev=np.nan,
    *,
    final_n_points=4,
    min_abs_slope_C_per_min=1e-4,
    max_extra_min_by_phase=None,
    allow_extrapolation=True,
):
    if max_extra_min_by_phase is None:
        max_extra_min_by_phase = {
            'heating': 60.0,
            'cooling': 30.0,
        }

    h = np.asarray(horizons, float)
    y = np.asarray(pred_T, float)

    m = np.isfinite(h) & np.isfinite(y)
    h = h[m]
    y = y[m]

    if len(h) == 0:
        return np.nan, 'no_valid_trajectory', np.nan

    o = np.argsort(h)
    h = h[o]
    y = y[o]

    h0 = np.r_[0.0, h]
    y0 = np.r_[float(current_T), y]

    if (
        crossed(phase, current_T, threshold) and
        direction_valid(phase, current_delta_prev)
    ):
        return 0.0, 'label_matched_at_source', float(h[-1])

    ph, pt = h0[0], y0[0]

    for hh, tt in zip(h0[1:], y0[1:]):
        d = tt - pt

        if (
            crossed(phase, tt, threshold) and
            direction_valid(phase, d)
        ):
            if not crossed(phase, pt, threshold):
                return (
                    linear_crossing(
                        ph, pt,
                        hh, tt,
                        threshold
                    ),
                    'label_matched_interpolated_crossing',
                    float(h[-1])
                )

            return (
                float(hh),
                'label_matched_direction_at_horizon',
                float(h[-1])
            )

        ph, pt = hh, tt

    if allow_extrapolation:
        n = min(int(final_n_points), len(h0))

        # Notebook 03 requires at least 3 points for final-trend fit.
        if n >= 3:
            slope, intercept = np.polyfit(
                h0[-n:],
                y0[-n:],
                1
            )

            good = (
                (
                    phase == 'heating' and
                    slope > float(min_abs_slope_C_per_min)
                ) or
                (
                    phase == 'cooling' and
                    slope < -float(min_abs_slope_C_per_min)
                )
            )

            if good:
                hp = (
                    float(threshold) - intercept
                ) / slope

                cap = float(
                    h0[-1] +
                    float(max_extra_min_by_phase[phase])
                )

                if np.isfinite(hp) and hp > h0[-1]:
                    return (
                        float(min(hp, cap)),
                        (
                            'label_matched_extrapolated_final_trend'
                            if hp <= cap
                            else 'projected_beyond_cap_used_cap'
                        ),
                        float(h[-1])
                    )

    return (
        float(h0[-1]),
        (
            'no_label_match_used_last_horizon'
            if allow_extrapolation
            else 'no_extrapolation_used_last_horizon'
        ),
        float(h[-1])
    )

def source_to_anchor_remaining(
    source_crossing_min,
    asof_gap_min
):
    if not np.isfinite(source_crossing_min):
        return np.nan

    return float(
        max(
            0.0,
            float(source_crossing_min) -
            float(asof_gap_min)
        )
    )

# Deterministic converter self-tests.
tests = [
    (
        'heating_interpolation',
        dict(
            phase='heating',
            current_T=21.0,
            threshold=21.6,
            horizons=[5, 10],
            pred_T=[21.4, 21.8],
            current_delta_prev=.1,
        ),
        7.5
    ),
    (
        'cooling_interpolation',
        dict(
            phase='cooling',
            current_T=26.0,
            threshold=25.4,
            horizons=[5, 10],
            pred_T=[25.7, 25.1],
            current_delta_prev=-.1,
        ),
        7.5
    ),
]

for name, kwargs, expected in tests:
    got = estimate_crossing_from_source_parametric(
        **kwargs,
        **CANONICAL_CONVERTER
    )[0]
    if abs(float(got) - expected) > 1e-10:
        raise RuntimeError(
            f'Converter self-test failed: {name}: {got} vs {expected}'
        )

print('Converter self-tests: PASSED')


## 4. Generate and cache temperature trajectories

Generate neural and Temperature ET forecasts once using Notebook 03's inference procedure. Reuse these forecasts for every extrapolation setting so that differences arise from duration calculation rather than model retraining or changed temperature forecasts.


In [ ]:


# Reproduce/cache Notebook-03 TEST trajectories

def horizon_grid(phase):
    return np.arange(
        STEP_MIN,
        INDIRECT_MAX_HORIZON_BY_PHASE[phase] + 1e-9,
        STEP_MIN,
        dtype=float
    )

def integrate_numpy_batch(
    h,
    pred_directed,
    T0,
    sign,
    v0_phys
):
    pred_directed = np.asarray(pred_directed, float)
    T0 = np.asarray(T0, float)
    sign = np.asarray(sign, float)
    v0_phys = np.asarray(v0_phys, float)

    pred_phys = sign[:, None] * pred_directed
    dt = np.r_[h[0], np.diff(h)].astype(float)

    prev = np.concatenate(
        [
            v0_phys[:, None],
            pred_phys[:, :-1]
        ],
        axis=1
    )

    return (
        T0[:, None] +
        np.cumsum(
            0.5 * dt[None, :] *
            (prev + pred_phys),
            axis=1
        )
    )

def initial_rate_array(df):
    if 'Room_dTdt_now_bwd2_C_per_min' in df.columns:
        v = pd.to_numeric(
            df['Room_dTdt_now_bwd2_C_per_min'],
            errors='coerce'
        ).to_numpy(float)
    else:
        v = np.full(len(df), np.nan)

    if 'Room_dTdt_central_-5min_C_per_min' in df.columns:
        fb = pd.to_numeric(
            df['Room_dTdt_central_-5min_C_per_min'],
            errors='coerce'
        ).to_numpy(float)
    else:
        fb = np.zeros(len(df), dtype=float)

    bad = ~np.isfinite(v)
    v[bad] = fb[bad]
    v[~np.isfinite(v)] = 0.0

    return v

def make_expanded_anchor_frame(chunk, h):
    H = len(h)

    expanded = (
        chunk.loc[
            chunk.index.repeat(H)
        ]
        .copy()
        .reset_index(drop=True)
    )

    expanded['horizon_min'] = np.tile(
        h,
        len(chunk)
    )

    expanded['target_timestamp'] = (
        pd.to_datetime(
            expanded['feature_source_timestamp']
        ) +
        pd.to_timedelta(
            expanded['horizon_min'],
            unit='m'
        )
    )

    return expanded

TRAJECTORY_CACHE_PATH = (
    CACHE_DIR /
    'NB06_frozen_TEST_predicted_trajectories.csv.gz'
)

if REUSE_TRAJECTORY_CACHE and TRAJECTORY_CACHE_PATH.exists():
    TRAJ = pd.read_csv(TRAJECTORY_CACHE_PATH)
    print('Loaded trajectory cache:', TRAJECTORY_CACHE_PATH)
else:
    trajectory_records = []
    chunk_size = 1024

    for phase in PHASES:
        ap = (
            EVAL_ANCHORS[
                EVAL_ANCHORS['phase'].eq(phase)
            ]
            .copy()
            .reset_index(drop=True)
        )

        h = horizon_grid(phase)
        H = len(h)

        print(
            f'{phase}: {len(ap)} TEST anchors × {H} horizons'
        )

        for lo in range(0, len(ap), chunk_size):
            chunk = (
                ap.iloc[lo:lo + chunk_size]
                .copy()
                .reset_index(drop=True)
            )
            B = len(chunk)

            expanded = make_expanded_anchor_frame(
                chunk,
                h
            )

            # Unified directed-rate NN.
            X = transform_nn(expanded)
            xt = torch.from_numpy(X).to(
                DEVICE,
                dtype=torch.float32,
                non_blocking=True
            )

            with torch.inference_mode():
                pred_directed = (
                    NN_MODEL(xt)
                    .float()
                    .reshape(B, H)
                    .cpu()
                    .numpy()
                )

            del xt

            sign_arr = pd.to_numeric(
                chunk['thermal_mode_sign'],
                errors='coerce'
            ).to_numpy(float)

            T0_arr = pd.to_numeric(
                chunk['Room_T'],
                errors='coerce'
            ).to_numpy(float)

            v0_arr = initial_rate_array(chunk)

            pT_nn = integrate_numpy_batch(
                h,
                pred_directed,
                T0_arr,
                sign_arr,
                v0_arr
            )

            # Room/mode temperature ET.
            pT_et = np.full(
                (B, H),
                np.nan,
                dtype=float
            )

            rooms = chunk['room'].astype(str).to_numpy()

            for room in np.unique(rooms):
                art = ET_ART.get((phase, str(room)))
                if art is None:
                    continue

                apos = np.flatnonzero(rooms == room)

                ridx = (
                    apos[:, None] * H +
                    np.arange(H)[None, :]
                ).reshape(-1)

                syn = expanded.iloc[ridx]

                Xe = art['imputer'].transform(
                    syn[art['feature_names']]
                    .apply(pd.to_numeric, errors='coerce')
                )

                delta = art['model'].predict(
                    Xe
                ).reshape(len(apos), H)

                pT_et[apos, :] = (
                    T0_arr[apos, None] +
                    delta
                )

            for bi in range(B):
                row = chunk.iloc[bi]

                common = {
                    'anchor_key': row['anchor_key'],
                    'phase': phase,
                    'room': row['room'],
                    'event_key': row['event_key'],
                    'anchor_timestamp': row['anchor_timestamp'],
                    'feature_source_timestamp': row['feature_source_timestamp'],
                    'asof_gap_min': row['asof_gap_min'],
                    'true_remaining_duration_min': row['true_remaining_duration_min'],
                    'Room_T': row['Room_T'],
                    'Phase_Target_T_C': row['Phase_Target_T_C'],
                    'thermal_mode_sign': row['thermal_mode_sign'],
                    'current_delta_prev_C': float(v0_arr[bi] * STEP_MIN),
                }

                for hi, hh in enumerate(h):
                    trajectory_records.append({
                        **common,
                        'horizon_min': float(hh),
                        'pred_T_Unified_Derivative_NN': float(pT_nn[bi, hi]),
                        'pred_T_Indirect_RoomPhase_ET': (
                            float(pT_et[bi, hi])
                            if np.isfinite(pT_et[bi, hi])
                            else np.nan
                        ),
                    })

    TRAJ = pd.DataFrame(trajectory_records)

    TRAJ.to_csv(
        TRAJECTORY_CACHE_PATH,
        index=False,
        compression='gzip'
    )

    print('Saved trajectory cache:', TRAJECTORY_CACHE_PATH)

# Basic cache integrity.
expected_anchor_count = EVAL_ANCHORS['anchor_key'].nunique()
if TRAJ['anchor_key'].nunique() != expected_anchor_count:
    raise RuntimeError(
        'Trajectory cache anchor coverage differs from authoritative TEST anchors.'
    )

print(
    'Trajectory cache:',
    len(TRAJ),
    'rows |',
    TRAJ['anchor_key'].nunique(),
    'anchors'
)


## 5. Reproduce reference durations

Apply the reference completion-time settings to the cached trajectories. Require agreement with Notebook 03's saved predictions for both trajectory methods before testing alternative settings.


In [ ]:


# Reference hard-converter replay

PRED_T_COL = {
    'Unified_Derivative_NN':
        'pred_T_Unified_Derivative_NN',
    'Indirect_RoomPhase_ET':
        'pred_T_Indirect_RoomPhase_ET',
}

def apply_converter_setting(
    trajectories,
    setting_name,
    setting,
):
    rows = []

    for anchor_key, g in trajectories.groupby(
        'anchor_key',
        sort=False
    ):
        g = g.sort_values('horizon_min')
        first = g.iloc[0]
        phase = str(first['phase'])

        h = pd.to_numeric(
            g['horizon_min'],
            errors='coerce'
        ).to_numpy(float)

        threshold = (
            float(first['Phase_Target_T_C']) -
            float(first['thermal_mode_sign']) *
            END_BAND_C
        )

        for method in TRAJECTORY_METHODS:
            y = pd.to_numeric(
                g[PRED_T_COL[method]],
                errors='coerce'
            ).to_numpy(float)

            if not np.isfinite(y).any():
                pred_anchor = np.nan
                cross = np.nan
                ctype = 'unsupported_no_room_phase_train_model'
                maxh = float(np.nanmax(h))
            else:
                cross, ctype, maxh = (
                    estimate_crossing_from_source_parametric(
                        phase=phase,
                        current_T=float(first['Room_T']),
                        threshold=threshold,
                        horizons=h,
                        pred_T=y,
                        current_delta_prev=float(
                            first['current_delta_prev_C']
                        ),
                        final_n_points=setting['final_n_points'],
                        min_abs_slope_C_per_min=
                            setting['min_abs_slope_C_per_min'],
                        max_extra_min_by_phase=
                            setting['max_extra_min_by_phase'],
                        allow_extrapolation=
                            setting['allow_extrapolation'],
                    )
                )

                pred_anchor = source_to_anchor_remaining(
                    cross,
                    float(first['asof_gap_min'])
                )

            rows.append({
                'setting': setting_name,
                'anchor_key': anchor_key,
                'phase': phase,
                'room': first['room'],
                'event_key': first['event_key'],
                'anchor_timestamp': first['anchor_timestamp'],
                'true_remaining_duration_min':
                    first['true_remaining_duration_min'],
                'method': method,
                'pred_remaining_duration_min':
                    pred_anchor,
                'crossing_from_feature_source_min':
                    cross,
                'crossing_type': ctype,
                'max_horizon_min': maxh,
            })

    return pd.DataFrame(rows)

CANONICAL_REPLAY = apply_converter_setting(
    TRAJ,
    'canonical',
    CANONICAL_CONVERTER
)

nb03_ref = NB03_TEST[
    NB03_TEST['method'].isin(
        TRAJECTORY_METHODS
    )
][
    [
        'anchor_key',
        'method',
        'pred_remaining_duration_min'
    ]
].copy()

check = CANONICAL_REPLAY.merge(
    nb03_ref,
    on=['anchor_key', 'method'],
    how='outer',
    suffixes=('_nb06', '_nb03'),
    validate='one_to_one'
)

if len(check) != len(nb03_ref):
    raise RuntimeError(
        'Notebook-06 canonical replay row count differs from Notebook 03.'
    )

replay_audit = []

for method in TRAJECTORY_METHODS:
    d = check[check['method'].eq(method)].copy()

    a = pd.to_numeric(
        d['pred_remaining_duration_min_nb06'],
        errors='coerce'
    )
    b = pd.to_numeric(
        d['pred_remaining_duration_min_nb03'],
        errors='coerce'
    )

    same_support = a.notna().equals(b.notna())

    finite = a.notna() & b.notna()

    max_abs_diff = (
        float(np.max(np.abs(a[finite] - b[finite])))
        if finite.any()
        else np.nan
    )

    replay_audit.append({
        'method': method,
        'same_finite_support': bool(same_support),
        'finite_predictions': int(finite.sum()),
        'max_abs_prediction_difference_min': max_abs_diff,
    })

REPLAY_AUDIT = pd.DataFrame(replay_audit)
REPLAY_AUDIT.to_csv(
    AUDIT_DIR / 'NB06_canonical_Notebook03_replay_audit.csv',
    index=False
)

display(REPLAY_AUDIT)

if STRICT_CANONICAL_REPLAY:
    if not REPLAY_AUDIT['same_finite_support'].all():
        raise RuntimeError(
            'Canonical replay finite-support mismatch.'
        )

    if (
        REPLAY_AUDIT[
            'max_abs_prediction_difference_min'
        ].fillna(0.0) >
        CANONICAL_REPLAY_TOL_MIN
    ).any():
        raise RuntimeError(
            'Canonical converter replay differs from Notebook 03. '
            'Do not run sensitivity on a non-identical reconstruction.'
        )

print('Canonical Notebook-03 converter replay: PASSED')


## 6. Define extrapolation perturbations

Vary one factor at a time around the reference setting. Include the reference calculation and a setting with extrapolation disabled. All outcomes are retained; test errors are not used to select a replacement.


In [ ]:


# One-factor-at-a-time converter sensitivity settings

CONVERTER_SETTINGS = {
    'canonical': {
        **CANONICAL_CONVERTER,
        'changed_factor': 'none',
    },

    # Number of final points used in linear extrapolation.
    'final_points_3': {
        **CANONICAL_CONVERTER,
        'final_n_points': 3,
        'changed_factor': 'final_n_points',
    },
    'final_points_5': {
        **CANONICAL_CONVERTER,
        'final_n_points': 5,
        'changed_factor': 'final_n_points',
    },

    # Minimum absolute final-trend slope.
    'min_slope_5e-5': {
        **CANONICAL_CONVERTER,
        'min_abs_slope_C_per_min': 5e-5,
        'changed_factor': 'min_abs_slope',
    },
    'min_slope_2e-4': {
        **CANONICAL_CONVERTER,
        'min_abs_slope_C_per_min': 2e-4,
        'changed_factor': 'min_abs_slope',
    },

    # Extrapolation cap.
    'short_caps': {
        **CANONICAL_CONVERTER,
        'max_extra_min_by_phase': {
            'heating': 45.0,
            'cooling': 20.0,
        },
        'changed_factor': 'max_extra_cap',
    },
    'long_caps': {
        **CANONICAL_CONVERTER,
        'max_extra_min_by_phase': {
            'heating': 90.0,
            'cooling': 45.0,
        },
        'changed_factor': 'max_extra_cap',
    },

    # Disable final-trend extrapolation entirely.
    'no_extrapolation': {
        **CANONICAL_CONVERTER,
        'allow_extrapolation': False,
        'changed_factor': 'extrapolation_on_off',
    },
}

settings_table = pd.DataFrame([
    {
        'setting': name,
        'changed_factor': cfg['changed_factor'],
        'final_n_points': cfg['final_n_points'],
        'min_abs_slope_C_per_min':
            cfg['min_abs_slope_C_per_min'],
        'heating_max_extra_min':
            cfg['max_extra_min_by_phase']['heating'],
        'cooling_max_extra_min':
            cfg['max_extra_min_by_phase']['cooling'],
        'allow_extrapolation':
            cfg['allow_extrapolation'],
    }
    for name, cfg in CONVERTER_SETTINGS.items()
])

settings_table.to_csv(
    TABLE_DIR / 'NB06_converter_sensitivity_settings.csv',
    index=False
)

display(settings_table)


In [ ]:


# Apply converter settings to fixed trajectories

converter_frames = []

for setting_name, setting in CONVERTER_SETTINGS.items():
    print('Converter sensitivity:', setting_name)
    converter_frames.append(
        apply_converter_setting(
            TRAJ,
            setting_name,
            setting
        )
    )

CONVERTER_PRED = pd.concat(
    converter_frames,
    ignore_index=True
)

CONVERTER_PRED.to_csv(
    PRED_DIR / 'NB06_converter_sensitivity_predictions.csv',
    index=False
)

# Primary sensitivity is evaluated on the paper's exact-common TEST support.
CONVERTER_COMMON = CONVERTER_PRED[
    CONVERTER_PRED['anchor_key']
    .astype(str)
    .isin(EXACT_COMMON_KEYS)
].copy()

expected_rows = (
    len(EXACT_COMMON_KEYS) *
    len(TRAJECTORY_METHODS) *
    len(CONVERTER_SETTINGS)
)

if len(CONVERTER_COMMON) != expected_rows:
    raise RuntimeError(
        f'Converter sensitivity row count mismatch: '
        f'{len(CONVERTER_COMMON)} vs expected {expected_rows}'
    )

if CONVERTER_COMMON[
    'pred_remaining_duration_min'
].isna().any():
    bad = CONVERTER_COMMON[
        CONVERTER_COMMON[
            'pred_remaining_duration_min'
        ].isna()
    ]
    raise RuntimeError(
        'Exact-common converter sensitivity contains missing predictions:\n' +
        bad.head(20).to_string(index=False)
    )

print('Converter sensitivity predictions complete.')


## 7. Compare duration predictions

For each method, operating mode and setting, report prediction shifts, completion outcomes and the paired error difference

\[
\Delta\mathrm{MAE}=\mathrm{MAE}_{\mathrm{setting}}-\mathrm{MAE}_{\mathrm{reference}}.
\]

Positive values indicate higher error under the perturbed setting. Paired event-bootstrap intervals describe variation within the fixed comparison cohort.


In [ ]:


# Converter metrics and prediction-shift audit

def duration_metrics(g):
    true = pd.to_numeric(
        g['true_remaining_duration_min'],
        errors='coerce'
    ).to_numpy(float)

    pred = pd.to_numeric(
        g['pred_remaining_duration_min'],
        errors='coerce'
    ).to_numpy(float)

    good = np.isfinite(true) & np.isfinite(pred)
    true = true[good]
    pred = pred[good]

    e = pred - true
    ae = np.abs(e)

    return pd.Series({
        'n': len(e),
        'MAE_min': float(np.mean(ae)),
        'RMSE_min': float(np.sqrt(np.mean(e ** 2))),
        'Bias_min': float(np.mean(e)),
        'MedianAE_min': float(np.median(ae)),
        'P90AE_min': float(np.percentile(ae, 90)),
    })

CONVERTER_METRICS = (
    CONVERTER_COMMON.groupby(
        ['setting', 'phase', 'method'],
        as_index=False
    )
    .apply(
        duration_metrics,
        include_groups=False
    )
    .reset_index(drop=True)
)

canonical_metric = (
    CONVERTER_METRICS[
        CONVERTER_METRICS['setting'].eq('canonical')
    ][
        ['phase', 'method', 'MAE_min']
    ]
    .rename(columns={
        'MAE_min': 'Canonical_MAE_min'
    })
)

CONVERTER_METRICS = CONVERTER_METRICS.merge(
    canonical_metric,
    on=['phase', 'method'],
    how='left',
    validate='many_to_one'
)

CONVERTER_METRICS['Delta_MAE_vs_canonical_min'] = (
    CONVERTER_METRICS['MAE_min'] -
    CONVERTER_METRICS['Canonical_MAE_min']
)

# Summarize changes in predicted remaining duration.
canonical_pred = (
    CONVERTER_COMMON[
        CONVERTER_COMMON['setting'].eq('canonical')
    ][
        [
            'anchor_key',
            'method',
            'pred_remaining_duration_min',
        ]
    ]
    .rename(columns={
        'pred_remaining_duration_min':
            'canonical_pred_min'
    })
)

shift = CONVERTER_COMMON.merge(
    canonical_pred,
    on=['anchor_key', 'method'],
    how='left',
    validate='many_to_one'
)

shift['prediction_shift_min'] = (
    shift['pred_remaining_duration_min'] -
    shift['canonical_pred_min']
)
shift['absolute_prediction_shift_min'] = (
    shift['prediction_shift_min'].abs()
)

SHIFT_SUMMARY = (
    shift.groupby(
        ['setting', 'phase', 'method'],
        as_index=False
    )
    .agg(
        changed_prediction_fraction=(
            'absolute_prediction_shift_min',
            lambda x: float(
                np.mean(
                    pd.to_numeric(
                        x,
                        errors='coerce'
                    ).to_numpy(float) > 1e-9
                )
            )
        ),
        mean_abs_prediction_shift_min=(
            'absolute_prediction_shift_min',
            'mean'
        ),
        median_abs_prediction_shift_min=(
            'absolute_prediction_shift_min',
            'median'
        ),
        p90_abs_prediction_shift_min=(
            'absolute_prediction_shift_min',
            lambda x: float(np.percentile(x, 90))
        ),
        max_abs_prediction_shift_min=(
            'absolute_prediction_shift_min',
            'max'
        ),
    )
)

CROSSING_COUNTS = (
    CONVERTER_COMMON.groupby(
        ['setting', 'phase', 'method', 'crossing_type'],
        as_index=False
    )
    .size()
)

CROSSING_COUNTS['fraction'] = (
    CROSSING_COUNTS['size'] /
    CROSSING_COUNTS.groupby(
        ['setting', 'phase', 'method']
    )['size'].transform('sum')
)

CONVERTER_METRICS.to_csv(
    TABLE_DIR / 'PAPER_NB06_converter_sensitivity_metrics.csv',
    index=False
)
SHIFT_SUMMARY.to_csv(
    TABLE_DIR / 'PAPER_NB06_converter_prediction_shift_summary.csv',
    index=False
)
CROSSING_COUNTS.to_csv(
    TABLE_DIR / 'PAPER_NB06_converter_crossing_type_counts.csv',
    index=False
)

print('Converter sensitivity metrics:')
display(
    CONVERTER_METRICS.sort_values(
        ['phase', 'method', 'setting']
    ).round(3)
)


In [ ]:


# Paired physical-event bootstrap for converter ΔMAE

def percentile_ci(x, ci_level=CI_LEVEL):
    a = np.asarray(
        pd.Series(x).dropna(),
        float
    )
    alpha = 1.0 - ci_level
    return (
        float(np.quantile(a, alpha / 2.0)),
        float(np.quantile(a, 1.0 - alpha / 2.0)),
    )

boot_rows = []

for phase in PHASES:
    for method in TRAJECTORY_METHODS:
        d = CONVERTER_COMMON[
            (CONVERTER_COMMON['phase'].eq(phase)) &
            (CONVERTER_COMMON['method'].eq(method))
        ].copy()

        w = d.pivot(
            index=[
                'anchor_key',
                'event_key',
                'true_remaining_duration_min'
            ],
            columns='setting',
            values='pred_remaining_duration_min'
        ).reset_index()

        if w[
            list(CONVERTER_SETTINGS)
        ].isna().any().any():
            raise RuntimeError(
                f'{phase}/{method}: converter bootstrap table has NaN.'
            )

        events = (
            w['event_key']
            .astype(str)
            .drop_duplicates()
            .tolist()
        )
        event_series = w['event_key'].astype(str)

        rows_by_event = {
            e: np.flatnonzero(
                event_series.eq(e).to_numpy()
            )
            for e in events
        }

        true_all = pd.to_numeric(
            w['true_remaining_duration_min'],
            errors='coerce'
        ).to_numpy(float)

        pred_arrays = {
            setting: pd.to_numeric(
                w[setting],
                errors='coerce'
            ).to_numpy(float)
            for setting in CONVERTER_SETTINGS
        }

        rng = np.random.default_rng(
            BOOTSTRAP_SEED +
            (0 if phase == 'heating' else 10000) +
            (0 if method == 'Unified_Derivative_NN' else 1000)
        )

        for b in range(N_BOOTSTRAP):
            sampled_events = rng.choice(
                events,
                size=len(events),
                replace=True
            )

            idx = np.concatenate([
                rows_by_event[e]
                for e in sampled_events
            ])

            true = true_all[idx]

            default_mae = float(
                np.mean(
                    np.abs(
                        pred_arrays['canonical'][idx] -
                        true
                    )
                )
            )

            for setting in CONVERTER_SETTINGS:
                if setting == 'canonical':
                    continue

                alt_mae = float(
                    np.mean(
                        np.abs(
                            pred_arrays[setting][idx] -
                            true
                        )
                    )
                )

                boot_rows.append({
                    'phase': phase,
                    'method': method,
                    'replicate': b,
                    'setting': setting,
                    'Canonical_MAE_min': default_mae,
                    'Alternative_MAE_min': alt_mae,
                    'Delta_MAE_Alternative_minus_Canonical_min':
                        alt_mae - default_mae,
                })

CONVERTER_BOOT = pd.DataFrame(boot_rows)

summary_rows = []

for (phase, method, setting), g in CONVERTER_BOOT.groupby(
    ['phase', 'method', 'setting'],
    sort=False
):
    delta = g[
        'Delta_MAE_Alternative_minus_Canonical_min'
    ].to_numpy(float)

    lo, hi = percentile_ci(delta)

    observed = float(
        CONVERTER_METRICS.loc[
            (CONVERTER_METRICS['phase'].eq(phase)) &
            (CONVERTER_METRICS['method'].eq(method)) &
            (CONVERTER_METRICS['setting'].eq(setting)),
            'Delta_MAE_vs_canonical_min'
        ].iloc[0]
    )

    summary_rows.append({
        'phase': phase,
        'method': method,
        'setting': setting,
        'observed_Delta_MAE_min': observed,
        'CI_level': CI_LEVEL,
        'CI_low': lo,
        'CI_high': hi,
        'bootstrap_mean_delta': float(np.mean(delta)),
        'bootstrap_fraction_delta_gt_0':
            float(np.mean(delta > 0.0)),
        'bootstrap_fraction_delta_lt_0':
            float(np.mean(delta < 0.0)),
    })

CONVERTER_BOOT_SUMMARY = pd.DataFrame(summary_rows)

CONVERTER_BOOT_SUMMARY.to_csv(
    TABLE_DIR / 'PAPER_NB06_converter_sensitivity_event_cluster_bootstrap_CI.csv',
    index=False
)

CONVERTER_BOOT.to_csv(
    TABLE_DIR / 'NB06_converter_sensitivity_bootstrap_replicates.csv.gz',
    index=False,
    compression='gzip'
)

print('Converter sensitivity paired event-cluster bootstrap:')
display(
    CONVERTER_BOOT_SUMMARY.round(3)
)


## 8. Reconstruct observations for endpoint analysis

Reproduce Notebook 01's event-detection preparation: parse timestamps as UTC, convert to local time, remove duplicate timestamps keeping the first record, invalidate readings at or below the specified limit, forward-fill and round to two decimals. Preserve original observation times rather than introducing an interpolated endpoint grid.


In [ ]:


# Exact Notebook-01 LocaliIN event-detection preparation

RAW_DATA_DIR = BASE_DIR / 'Datasets'
DRDADRAS_DIR = BASE_DIR / private_value('inputs.literal_001')

LOCALI_CANDIDATES = [
    RAW_DATA_DIR / private_value('inputs.literal_005'),
    DRDADRAS_DIR / private_value('inputs.literal_005'),
]

LOCALI_FILE = next(
    (p for p in LOCALI_CANDIDATES if p.exists()),
    None
)

if LOCALI_FILE is None:
    raise FileNotFoundError(
        private_value('inputs.literal_023') +
        '\n'.join(map(str, LOCALI_CANDIDATES))
    )

def read_csv_indexed_dadras_style(path, label):
    d = pd.read_csv(path)
    d.columns = d.columns.astype(str).str.strip()

    if 'timestamp' not in d.columns:
        raise ValueError(
            f'{label} has no timestamp column: {path}'
        )

    raw = d['timestamp'].astype(str).str.strip()

    try:
        ts = pd.to_datetime(
            raw,
            errors='coerce',
            utc=True,
            format='mixed',
            dayfirst=False
        )
    except TypeError:
        ts = pd.to_datetime(
            raw,
            errors='coerce',
            utc=True,
            dayfirst=False,
            infer_datetime_format=True
        )

    failed = ts.isna()

    if failed.any():
        try:
            fallback = pd.to_datetime(
                raw[failed],
                errors='coerce',
                utc=True,
                format='mixed',
                dayfirst=True
            )
        except TypeError:
            fallback = pd.to_datetime(
                raw[failed],
                errors='coerce',
                utc=True,
                dayfirst=True,
                infer_datetime_format=True
            )

        ts.loc[failed] = fallback

    ts = (
        ts.dt
        .tz_convert('Europe/Rome')
        .dt
        .tz_localize(None)
    )

    d = d.drop(columns=['timestamp'])
    d.index = ts
    d = d[~d.index.isna()]
    d = d.sort_index()
    d = d[~d.index.duplicated(keep='first')]

    return d

raw_locali = read_csv_indexed_dadras_style(
    LOCALI_FILE,
    private_value('inputs.literal_005')
)

# Exact Notebook-01 event-detection cleaning:
# where(values > -50, NaN).ffill().round(2)
raw_locali = raw_locali.apply(
    pd.to_numeric,
    errors='coerce'
)

locali_detection = (
    raw_locali
    .where(raw_locali > -50, np.nan)
    .ffill()
    .round(decimals=2)
    .reset_index()
    .rename(columns={'index': 'timestamp'})
)

print('LocaliIN source:', LOCALI_FILE)
print(
    'Detection frame:',
    len(locali_detection),
    'rows |',
    locali_detection['timestamp'].min(),
    'to',
    locali_detection['timestamp'].max()
)


In [ ]:


# Load saved event tables

event_frames = []

for phase in PHASES:
    p = DATA_DIR / phase / f'{phase}_events_5min_grid.csv'
    if not p.exists():
        raise FileNotFoundError(
            f'Authoritative Notebook-01 event table missing: {p}'
        )

    ev = pd.read_csv(p)
    ev['phase'] = phase

    for c in ['start_time', 'end_time']:
        if c not in ev.columns:
            raise RuntimeError(
                f'{phase}: event table missing {c}'
            )
        ev[c] = pd.to_datetime(
            ev[c],
            errors='coerce'
        )

    ev['event_id'] = ev['event_id'].map(norm_event_id)
    ev['room'] = ev['room'].astype(str).str.strip()
    ev['event_key'] = (
        ev['phase'] + '|' +
        ev['room'] + '|' +
        ev['event_id']
    )

    ev['duration_min'] = pd.to_numeric(
        ev['duration_min'],
        errors='coerce'
    )

    event_frames.append(ev)

EVENTS = pd.concat(
    event_frames,
    ignore_index=True,
    sort=False
)

EXPECTED_EVENT_COUNTS = private_value('nb06.expected_event_counts')

event_counts = (
    EVENTS.groupby('phase')
    .agg(
        events=('event_key', 'nunique'),
        rooms=('room', 'nunique'),
    )
    .reset_index()
)

display(event_counts)

for _, r in event_counts.iterrows():
    phase = r['phase']
    if int(r['events']) != EXPECTED_EVENT_COUNTS[phase]:
        raise RuntimeError(
            f'{phase}: authoritative event count changed: '
            f'{int(r["events"])} vs expected '
            f'{EXPECTED_EVENT_COUNTS[phase]}.'
        )

print('Authoritative event-count audit: PASSED')


### Fixed starts and cohort

For each saved event, retain its start and use the cleaned room observations to find the first subsequent row satisfying the alternative temperature allowance and required direction. Keep the original daily search window. Record when an alternative endpoint cannot be recovered.


In [ ]:


# Recompute event ends for 0.3 / 0.4 / 0.5 C bands

def parse_prefixes(event_row):
    if 'prefixes' in event_row.index:
        v = event_row.get('prefixes')
        if pd.notna(v) and str(v).strip():
            prefixes = [
                x.strip()
                for x in str(v).split(',')
                if x.strip()
            ]
            if prefixes:
                return prefixes

    return [str(event_row['room']).strip()]

def room_detection_frame(prefixes):
    T_cols = [
        f'{p}_T'
        for p in prefixes
        if f'{p}_T' in locali_detection.columns
    ]
    V_cols = [
        f'{p}_V'
        for p in prefixes
        if f'{p}_V' in locali_detection.columns
    ]

    if not T_cols or not V_cols:
        raise RuntimeError(
            f'Missing LocaliIN T/V columns for prefixes {prefixes}. '
            f'T={T_cols}, V={V_cols}'
        )

    out = locali_detection[
        ['timestamp'] + T_cols + V_cols
    ].copy()

    out['Room_T'] = (
        out[T_cols]
        .apply(pd.to_numeric, errors='coerce')
        .mean(axis=1)
    )

    out['Room_V'] = (
        out[V_cols]
        .apply(pd.to_numeric, errors='coerce')
        .mean(axis=1)
    )

    out = (
        out.dropna(
            subset=['timestamp', 'Room_T', 'Room_V']
        )
        .sort_values('timestamp')
        .drop_duplicates(
            subset=['timestamp'],
            keep='first'
        )
        .reset_index(drop=True)
    )

    out['delta_from_lag_C'] = (
        out['Room_T'] -
        out['Room_T'].shift(1)
    )

    return out

def clock_window_mask(ts, start_clock, end_clock):
    """Reproduce Notebook 01 clock-window comparisons using hour and minute only.
    Seconds are intentionally ignored, including at an inclusive endpoint.
    """
    ts = pd.to_datetime(ts, errors='coerce')

    # Match Notebook 01 clock comparisons using hour and minute only; ignore seconds.
    minutes = ts.dt.hour * 60 + ts.dt.minute

    def parse_clock(x):
        hh, mm = str(x).split(':')[:2]
        return int(hh) * 60 + int(mm)

    lo = parse_clock(start_clock)
    hi = parse_clock(end_clock)

    return (
        (minutes >= lo) &
        (minutes <= hi)
    )

ROOM_DETECTION_CACHE = {}

def get_room_detection(prefixes):
    key = tuple(prefixes)
    if key not in ROOM_DETECTION_CACHE:
        ROOM_DETECTION_CACHE[key] = room_detection_frame(prefixes)
    return ROOM_DETECTION_CACHE[key]

def event_target_from_saved_event(row):
    phase = str(row['phase'])

    if phase == 'cooling':
        # Reference retrospective cooling target.
        return 25.0

    for c in [
        'target_at_start',
        'target_at_end',
        'Heating_Target_for_detection'
    ]:
        if c in row.index:
            v = pd.to_numeric(
                row.get(c),
                errors='coerce'
            )
            if np.isfinite(v):
                return float(v)

    raise RuntimeError(
        f'Heating event {row["event_key"]}: no saved fixed target.'
    )

def recompute_end_for_saved_event(row, end_band_C):
    phase = str(row['phase'])
    start = pd.Timestamp(row['start_time'])
    prefixes = parse_prefixes(row)
    d = get_room_detection(prefixes)

    target = event_target_from_saved_event(row)

    same_day = (
        pd.to_datetime(d['timestamp']).dt.date ==
        start.date()
    )

    w0, w1 = PHASE_END_SEARCH_WINDOWS[phase]
    in_window = clock_window_mask(
        d['timestamp'],
        w0,
        w1
    )

    after_start = (
        pd.to_datetime(d['timestamp']) >
        start
    )

    if phase == 'heating':
        threshold = target - float(end_band_C)
        end_cond = (
            (d['delta_from_lag_C'] > 0) &
            (d['Room_T'] >= threshold)
        )
    else:
        threshold = target + float(end_band_C)
        end_cond = (
            (d['delta_from_lag_C'] < 0) &
            (d['Room_T'] <= threshold)
        )

    cand = d[
        same_day &
        in_window &
        after_start &
        end_cond.fillna(False)
    ].copy()

    if cand.empty:
        return {
            'alternative_end_time': pd.NaT,
            'alternative_duration_min': np.nan,
            'alternative_end_temperature_C': np.nan,
            'alternative_threshold_C': threshold,
            'end_found': False,
        }

    first = cand.iloc[0]
    end = pd.Timestamp(first['timestamp'])

    # Notebook 01's duration uses round((End-Start).seconds/60, 0).
    duration = round(
        (end - start).seconds / 60.0,
        0
    )

    return {
        'alternative_end_time': end,
        'alternative_duration_min': float(duration),
        'alternative_end_temperature_C': float(first['Room_T']),
        'alternative_threshold_C': float(threshold),
        'end_found': True,
    }

event_sensitivity_rows = []

for i, row in EVENTS.iterrows():
    if (i + 1) % 250 == 0:
        print(f'Endpoint sensitivity: {i+1}/{len(EVENTS)} events')

    for band in END_BANDS_C:
        result = recompute_end_for_saved_event(
            row,
            band
        )

        alt_end = result['alternative_end_time']
        official_end = pd.Timestamp(row['end_time'])

        signed_shift = (
            (alt_end - official_end).total_seconds() / 60.0
            if pd.notna(alt_end)
            else np.nan
        )

        alt_duration = result['alternative_duration_min']
        cap = MAX_RAMPUP_MINUTES_BY_PHASE[str(row['phase'])]

        event_sensitivity_rows.append({
            'event_key': row['event_key'],
            'phase': row['phase'],
            'room': row['room'],
            'event_id': row['event_id'],
            'start_time': row['start_time'],
            'official_end_time': official_end,
            'official_duration_min': row['duration_min'],
            'end_band_C': float(band),
            **result,
            'signed_end_shift_min': signed_shift,
            'absolute_end_shift_min': (
                abs(signed_shift)
                if np.isfinite(signed_shift)
                else np.nan
            ),
            'same_end_timestamp': bool(
                pd.notna(alt_end) and
                alt_end == official_end
            ),
            'within_original_duration_scope': bool(
                np.isfinite(alt_duration) and
                MIN_RAMPUP_MINUTES <= alt_duration <= cap
            ),
        })

EVENT_END_SENS = pd.DataFrame(
    event_sensitivity_rows
)

EVENT_END_SENS.to_csv(
    TABLE_DIR / 'NB06_event_end_band_sensitivity_event_level.csv',
    index=False
)

# Require the default endpoint allowance to reproduce every saved event end.

default_end = EVENT_END_SENS[
    EVENT_END_SENS['end_band_C'].eq(
        CANONICAL_EVENT_END_BAND_C
    )
].copy()

default_end['duration_diff_min'] = (
    pd.to_numeric(
        default_end['alternative_duration_min'],
        errors='coerce'
    ) -
    pd.to_numeric(
        default_end['official_duration_min'],
        errors='coerce'
    )
)

mismatch = default_end[
    (~default_end['end_found']) |
    (~default_end['same_end_timestamp']) |
    (default_end['duration_diff_min'].abs() > 1e-12)
].copy()

mismatch.to_csv(
    AUDIT_DIR / 'NB06_BAD_default_0p4_event_end_reproduction_mismatches.csv',
    index=False
)

reproduction_summary = (
    default_end.groupby('phase')
    .agg(
        events=('event_key', 'nunique'),
        end_found=('end_found', 'sum'),
        exact_same_end=('same_end_timestamp', 'sum'),
        max_abs_duration_diff_min=(
            'duration_diff_min',
            lambda x: float(
                np.nanmax(
                    np.abs(
                        pd.to_numeric(
                            x,
                            errors='coerce'
                        ).to_numpy(float)
                    )
                )
            )
        ),
    )
    .reset_index()
)

reproduction_summary.to_csv(
    AUDIT_DIR / 'NB06_default_0p4_event_end_reproduction_summary.csv',
    index=False
)

display(reproduction_summary)

if STRICT_EVENT_END_REPRODUCTION and len(mismatch):
    raise RuntimeError(
        f'Default 0.4 C endpoint reconstruction failed for {len(mismatch)} '
        'authoritative events. Sensitivity results are invalid until '
        'Notebook-01 event detection is reproduced exactly. '
        'See the mismatch audit CSV.'
    )

print('Canonical 0.4 C event-end reconstruction: PASSED')


## 9. Summarize endpoint changes

Compare recovered endpoints with the reference endpoint using signed and absolute time shifts, unchanged-endpoint rates and timing bands. Report recovery counts alongside shift summaries: statistics over recovered endpoints do not represent events for which no alternative endpoint was found.


In [ ]:


# Event-end sensitivity summary + event bootstrap intervals

summary_rows = []

for (phase, band), g in EVENT_END_SENS.groupby(
    ['phase', 'end_band_C'],
    sort=False
):
    shift = pd.to_numeric(
        g['signed_end_shift_min'],
        errors='coerce'
    )
    abs_shift = shift.abs()

    summary_rows.append({
        'phase': phase,
        'end_band_C': band,
        'events': int(g['event_key'].nunique()),
        'end_found_pct': float(
            100.0 * g['end_found'].mean()
        ),
        'same_end_timestamp_pct': float(
            100.0 * g['same_end_timestamp'].mean()
        ),
        'mean_signed_end_shift_min': float(
            shift.mean()
        ),
        'median_signed_end_shift_min': float(
            shift.median()
        ),
        'mean_abs_end_shift_min': float(
            abs_shift.mean()
        ),
        'median_abs_end_shift_min': float(
            abs_shift.median()
        ),
        'P90_abs_end_shift_min': float(
            abs_shift.quantile(.90)
        ),
        'within_5min_of_default_pct': float(
            100.0 * (abs_shift <= 5.0).mean()
        ),
        'within_10min_of_default_pct': float(
            100.0 * (abs_shift <= 10.0).mean()
        ),
        'within_original_duration_scope_pct': float(
            100.0 *
            g['within_original_duration_scope'].mean()
        ),
    })

EVENT_END_SUMMARY = pd.DataFrame(summary_rows)

# Event bootstrap for mean absolute endpoint shift.
boot_rows = []

for phase in PHASES:
    d = EVENT_END_SENS[
        EVENT_END_SENS['phase'].eq(phase)
    ].copy()

    events = d['event_key'].drop_duplicates().tolist()

    wide_shift = d.pivot(
        index='event_key',
        columns='end_band_C',
        values='absolute_end_shift_min'
    )

    rng = np.random.default_rng(
        BOOTSTRAP_SEED +
        (0 if phase == 'heating' else 10000)
    )

    for band in END_BANDS_C:
        if band == CANONICAL_EVENT_END_BAND_C:
            continue

        values = pd.to_numeric(
            wide_shift[band],
            errors='coerce'
        )

        for b in range(N_BOOTSTRAP):
            sampled = rng.choice(
                events,
                size=len(events),
                replace=True
            )

            x = values.reindex(
                sampled
            ).to_numpy(float)

            boot_rows.append({
                'phase': phase,
                'end_band_C': float(band),
                'replicate': b,
                'mean_abs_end_shift_min':
                    float(np.nanmean(x)),
            })

END_BOOT = pd.DataFrame(boot_rows)

end_ci_rows = []

for (phase, band), g in END_BOOT.groupby(
    ['phase', 'end_band_C'],
    sort=False
):
    lo, hi = percentile_ci(
        g['mean_abs_end_shift_min']
    )

    observed = float(
        EVENT_END_SUMMARY.loc[
            (EVENT_END_SUMMARY['phase'].eq(phase)) &
            (EVENT_END_SUMMARY['end_band_C'].eq(band)),
            'mean_abs_end_shift_min'
        ].iloc[0]
    )

    end_ci_rows.append({
        'phase': phase,
        'end_band_C': band,
        'mean_abs_end_shift_min': observed,
        'CI_level': CI_LEVEL,
        'CI_low': lo,
        'CI_high': hi,
    })

EVENT_END_BOOTSTRAP = pd.DataFrame(
    end_ci_rows
)

EVENT_END_SUMMARY.to_csv(
    TABLE_DIR / 'PAPER_NB06_event_end_band_sensitivity_summary.csv',
    index=False
)

EVENT_END_BOOTSTRAP.to_csv(
    TABLE_DIR / 'PAPER_NB06_event_end_band_sensitivity_bootstrap_CI.csv',
    index=False
)

END_BOOT.to_csv(
    TABLE_DIR / 'NB06_event_end_band_bootstrap_replicates.csv.gz',
    index=False,
    compression='gzip'
)

print('Event-end construct sensitivity:')
display(
    EVENT_END_SUMMARY.round(3)
)

print('\nBootstrap CI for mean absolute endpoint shift:')
display(
    EVENT_END_BOOTSTRAP.round(3)
)


## 10. Plot sensitivity results

Visualize duration-error changes under extrapolation perturbations and endpoint shifts under alternative label definitions. Keep these analyses separate because they vary different parts of the prediction problem.


In [ ]:


# Paper-ready robustness figures

# Figure A — converter ΔMAE relative to reference

plot_settings = [
    s for s in CONVERTER_SETTINGS
    if s != 'canonical'
]

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 9),
    constrained_layout=True
)

for row_i, phase in enumerate(PHASES):
    for col_i, method in enumerate(TRAJECTORY_METHODS):
        ax = axes[row_i, col_i]

        d = (
            CONVERTER_BOOT_SUMMARY[
                (CONVERTER_BOOT_SUMMARY['phase'].eq(phase)) &
                (CONVERTER_BOOT_SUMMARY['method'].eq(method))
            ]
            .set_index('setting')
            .reindex(plot_settings)
            .reset_index()
        )

        y = np.arange(len(d))
        center = d['observed_Delta_MAE_min'].to_numpy(float)
        lo = d['CI_low'].to_numpy(float)
        hi = d['CI_high'].to_numpy(float)

        ax.errorbar(
            center,
            y,
            xerr=np.vstack([
                center - lo,
                hi - center
            ]),
            fmt='o',
            capsize=4
        )
        ax.axvline(0.0, linestyle='--')

        ax.set_yticks(
            y,
            d['setting']
        )
        ax.set_xlabel(
            'ΔMAE vs canonical converter (min)'
        )
        ax.set_title(
            f'{phase.capitalize()} — '
            f'{"Unified NN" if method=="Unified_Derivative_NN" else "Indirect ET"}'
        )
        ax.grid(axis='x', alpha=.25)

fig.savefig(
    FIG_DIR / 'PAPER_NB06_converter_sensitivity_delta_MAE.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

# Figure B — event-end shift distributions

fig, axes = plt.subplots(
    1, 2,
    figsize=(11.5, 4.8),
    constrained_layout=True
)

for ax, phase in zip(axes, PHASES):
    d = EVENT_END_SENS[
        EVENT_END_SENS['phase'].eq(phase) &
        EVENT_END_SENS['end_band_C'].isin([0.3, 0.5])
    ].copy()

    data = [
        pd.to_numeric(
            d.loc[
                d['end_band_C'].eq(band),
                'signed_end_shift_min'
            ],
            errors='coerce'
        ).dropna().to_numpy(float)
        for band in [0.3, 0.5]
    ]

    ax.boxplot(
        data,
        tick_labels=['0.3°C', '0.5°C'],
        showfliers=False
    )
    ax.axhline(0.0, linestyle='--')
    ax.set_ylabel(
        'Alternative end − canonical 0.4°C end (min)'
    )
    ax.set_title(
        phase.capitalize()
    )
    ax.grid(axis='y', alpha=.25)

fig.savefig(
    FIG_DIR / 'PAPER_NB06_event_end_band_signed_shift.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

# Figure C — fraction of events keeping identical end timestamp

same = EVENT_END_SUMMARY[
    EVENT_END_SUMMARY['end_band_C'].isin(
        END_BANDS_C
    )
].copy()

fig, ax = plt.subplots(
    figsize=(8.5, 4.8)
)

x = np.arange(len(END_BANDS_C))
width = 0.35

for j, phase in enumerate(PHASES):
    d = (
        same[same['phase'].eq(phase)]
        .set_index('end_band_C')
        .reindex(END_BANDS_C)
    )

    ax.bar(
        x + (j - .5) * width,
        d['same_end_timestamp_pct'],
        width=width,
        label=phase.capitalize()
    )

ax.set_xticks(
    x,
    [f'{x:.1f}°C' for x in END_BANDS_C]
)
ax.set_ylim(0, 105)
ax.set_ylabel(
    'Events with identical end timestamp (%)'
)
ax.set_xlabel(
    'End-neighborhood band'
)
ax.set_title(
    'Construct sensitivity of operational event end'
)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=.25)

fig.tight_layout()
fig.savefig(
    FIG_DIR / 'PAPER_NB06_event_end_identical_fraction.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

print('Figures saved to:', FIG_DIR)


## 11. Save analysis provenance

Record the input run, split fingerprint, trained artifacts, reference prediction checks, default endpoint checks, tested settings and output paths. The manifest documents the fixed models and data used for these sensitivity analyses.


In [ ]:


# Final integrity checks + manifest

status_rows = [
    {
        'check': 'Uses PAPER-TEMP rather than PAPER-TEMP-NEW',
        'status': 'PASS',
    },
    {
        'check': 'Shared split matches Notebook 02 and Notebook 03',
        'status': 'PASS',
    },
    {
        'check': 'Canonical NN and temperature ET artifacts are frozen',
        'status': 'PASS',
    },
    {
        'check': 'Notebook-03 canonical hard predictions replayed',
        'status': 'PASS',
    },
    {
        'check': 'Converter alternatives change post-processing only',
        'status': 'PASS',
    },
    {
        'check': 'Raw LocaliIN prepared with Notebook-01 event-detection cleaning',
        'status': 'PASS',
    },
    {
        'check': 'Canonical 0.4 C event ends reproduced exactly',
        'status': 'PASS',
    },
    {
        'check': 'Event starts/cohort held fixed in event-end sensitivity',
        'status': 'PASS',
    },
    {
        'check': 'No TEST-based setting selection performed',
        'status': 'PASS',
    },
]

STATUS = pd.DataFrame(status_rows)
STATUS.to_csv(
    AUDIT_DIR / 'NB06_final_integrity_checks.csv',
    index=False
)

manifest = {
    'notebook':
        '06_converter_and_event_end_sensitivity_PAPER_FINAL',
    'role':
        'post-hoc robustness and event-end construct-validity analysis; no retraining',
    'base_dir':
        str(BASE_DIR),
    'uses_balanced_final':
        False,
    'shared_split_sha256':
        CURRENT_SPLIT_SHA256,
    'canonical_training_run':
        str(TRAIN_RUN),
    'canonical_notebook03_run':
        str(NB03_RUN),
    'canonical_converter':
        CANONICAL_CONVERTER,
    'converter_sensitivity_settings':
        CONVERTER_SETTINGS,
    'event_end_bands_C':
        END_BANDS_C,
    'canonical_event_end_band_C':
        CANONICAL_EVENT_END_BAND_C,
    'event_end_sensitivity_policy':
        (
            'fixed authoritative event starts and cohort; '
            'first phase-direction-valid end recomputed from exact '
            'Notebook-01 LocaliIN detection preparation'
        ),
    'raw_localiin_file':
        str(LOCALI_FILE),
    'canonical_replay_audit':
        REPLAY_AUDIT.to_dict(orient='records'),
    'default_event_end_reproduction':
        reproduction_summary.to_dict(orient='records'),
    'bootstrap': {
        'unit':
            'physical event',
        'replicates':
            N_BOOTSTRAP,
        'CI_level':
            CI_LEVEL,
    },
    'test_used_for_selection':
        False,
    'interpretation_guardrail':
        (
            'Sensitivity settings are descriptive robustness checks. '
            'They do not replace the canonical converter/event definition '
            'and are not selected on TEST performance.'
        ),
    'key_outputs': {
        'converter_metrics':
            str(
                TABLE_DIR /
                'PAPER_NB06_converter_sensitivity_metrics.csv'
            ),
        'converter_bootstrap':
            str(
                TABLE_DIR /
                'PAPER_NB06_converter_sensitivity_event_cluster_bootstrap_CI.csv'
            ),
        'converter_crossing_counts':
            str(
                TABLE_DIR /
                'PAPER_NB06_converter_crossing_type_counts.csv'
            ),
        'event_end_event_level':
            str(
                TABLE_DIR /
                'NB06_event_end_band_sensitivity_event_level.csv'
            ),
        'event_end_summary':
            str(
                TABLE_DIR /
                'PAPER_NB06_event_end_band_sensitivity_summary.csv'
            ),
        'event_end_bootstrap':
            str(
                TABLE_DIR /
                'PAPER_NB06_event_end_band_sensitivity_bootstrap_CI.csv'
            ),
    },
}

with open(
    OUT_ROOT / 'NB06_sensitivity_manifest.json',
    'w',
    encoding='utf-8'
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        default=str
    )

print('\nFINAL NOTEBOOK-06 INTEGRITY STATUS')
display(STATUS)

print('\nOutput root:', OUT_ROOT)


## Interpretation

Prediction stability applies to the tested extrapolation settings; it does not establish insensitivity to every duration-calculation rule. Interpret neural and Temperature ET results separately by operating mode.

Endpoint sensitivity describes the dependence of an operational label on its definition. Report missing alternative endpoints and the population used for shift summaries. These results do not show whether model rankings persist under alternative labels, because predictive accuracy is not recomputed against those labels.
